In [9]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

In [2]:
# Load Datasets

df = pd.read_csv('cleaned_transcripts.csv')   # video_id, title, datetime, transcript
queries = pd.read_csv('search_queries.csv')         # query, relevant_video_id

print(f"Videos  : {len(df)}")
print(f"Queries : {len(queries)}")
df.head()

Videos  : 123
Queries : 83


,video_id,title,datetime,transcript
0,2KaAHD2TOos,Is dust really people?,2026-02-13,(finger squeaking) - Is dust just people? You ...
1,iT2n41ZFDc0,The Intelligence Test Where Ants Beat Humans,2026-02-05,- Thank you to AnyDesk for supporting PBS. Eve...
2,UI8pp2j2fZ8,What time is it on Voyager 1? 🤔,2026-01-30,- What's the farthest manmade object from Eart...
3,rTvTsvT7y7s,What& 39;s REALLY killing all the birds?,2026-01-23,"- You may have gotten wind, the turbines like ..."
4,AxE13l_rXOI,How much does Will Smith know about nature?,2026-01-18,We recently had the opportunity to play a fun ...


In [3]:
# Load 3 Models

model_names = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
    'multi-qa-MiniLM-L6-cos-v1'
]

In [4]:
# Loop over each model

results = []

for model_name in model_names:
    print(f"\nProcessing: {model_name}")
    model = SentenceTransformer(model_name)

    # Generate embeddings for titles and transcripts
    title_embeddings = model.encode(df['title'].tolist(), show_progress_bar=True)
    transcript_embeddings = model.encode(df['transcript'].tolist(), show_progress_bar=True)

    # Generate embeddings for queries
    query_embeddings = model.encode(queries['query'].tolist(), show_progress_bar=True)

    # Test all metrics
    metrics = {
        'Cosine': lambda a, b: cosine_similarity(a, b),
        'DotProduct': lambda a, b: np.dot(a, b.T),
        'Euclidean': lambda a, b: -cdist(a, b, metric='euclidean'),  # negative = higher is closer
        'Manhattan': lambda a, b: -cdist(a, b, metric='cityblock'),
        'Chebyshev': lambda a, b: -cdist(a, b, metric='chebyshev'),
    }

    for metric_name, metric_fn in metrics.items():
        for field, embeddings in [('title', title_embeddings), ('transcript', transcript_embeddings)]:
            top1 = top3 = top5 = 0
            rank_list = []

            # Rank videos for each query
            scores = metric_fn(query_embeddings, embeddings)  # shape: (n_queries, n_videos)

            for i, row in queries.iterrows():
                expected_id = row['relevant_video_id']
                ranked_ids = df['video_id'].iloc[np.argsort(scores[i])[::-1]].tolist()

                rank = ranked_ids.index(expected_id) + 1 if expected_id in ranked_ids else len(ranked_ids) + 1

                # Evaluate
                if rank == 1: top1 += 1
                if rank <= 3: top3 += 1
                if rank <= 5: top5 += 1
                rank_list.append(rank)

            n = len(queries)
            results.append({
                'Model': model_name,
                'Field': field,
                'Metric': metric_name,
                'Top-1 Recall': round(top1 / n, 3),
                'Top-3 Recall': round(top3 / n, 3),
                'Top-5 Recall': round(top5 / n, 3),
                'Avg Rank': round(np.mean(rank_list), 2)
            })


Processing: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2664.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 14.48it/s]



Processing: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2263.31it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00,  3.50it/s]



Processing: multi-qa-MiniLM-L6-cos-v1


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2315.38it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 21.95it/s]


In [5]:
# Comparison Table

results_df = pd.DataFrame(results)
results_df.to_csv('embedding_evaluation_results.csv', index=False)
results_df

,Model,Field,Metric,Top-1 Recall,Top-3 Recall,Top-5 Recall,Avg Rank
0,all-MiniLM-L6-v2,title,Cosine,0.964,0.976,0.976,2.55
1,all-MiniLM-L6-v2,transcript,Cosine,0.855,0.964,0.976,2.83
2,all-MiniLM-L6-v2,title,DotProduct,0.964,0.976,0.976,2.55
3,all-MiniLM-L6-v2,transcript,DotProduct,0.855,0.964,0.976,2.83
4,all-MiniLM-L6-v2,title,Euclidean,0.964,0.976,0.976,2.55
5,all-MiniLM-L6-v2,transcript,Euclidean,0.855,0.964,0.976,2.83
6,all-MiniLM-L6-v2,title,Manhattan,0.964,0.976,0.976,2.59
7,all-MiniLM-L6-v2,transcript,Manhattan,0.843,0.976,0.976,2.93
8,all-MiniLM-L6-v2,title,Chebyshev,0.952,0.964,0.964,2.87
9,all-MiniLM-L6-v2,transcript,Chebyshev,0.614,0.783,0.843,6.55


In [7]:
# Best Model & Metric

best = results_df.sort_values(
    by=['Top-3 Recall', 'Top-1 Recall', 'Avg Rank'],
    ascending=[False, False, True]
).iloc[0]

In [8]:
print(f"""
===== BEST COMBINATION =====
Model  : {best['Model']}
Field  : {best['Field']}
Metric : {best['Metric']}
Top-1  : {best['Top-1 Recall']}
Top-3  : {best['Top-3 Recall']}
Avg Rank: {best['Avg Rank']}
""")


===== BEST COMBINATION =====
Model  : all-mpnet-base-v2
Field  : title
Metric : Cosine
Top-1  : 0.976
Top-3  : 0.988
Avg Rank: 2.49

